In [2]:
import sys
sys.path.append('../backend')

from db.models import SessionLocal, Problem

db       = SessionLocal()
problems = db.query(Problem).all()
db.close()

print(f"Total cached problems: {len(problems)}")
print()

# Check rating distribution
ratings = [p.rating for p in problems if p.rating and p.rating > 0]
print(f"Problems with ratings: {len(ratings)}")
print(f"Rating range: {min(ratings)} – {max(ratings)}")
print()

# Check tag distribution
from collections import Counter
all_tags = []
for p in problems:
    if p.tags:
        all_tags.extend(p.tags)

tag_counts = Counter(all_tags)
print("Top 20 tags by problem count:")
for tag, count in tag_counts.most_common(20):
    bar = '█' * (count // 100)
    print(f"  {tag:30s}: {count:5d}  {bar}")

Total cached problems: 11255

Problems with ratings: 10979
Rating range: 800 – 3500

Top 20 tags by problem count:
  greedy                        :  3504  ███████████████████████████████████
  math                          :  3436  ██████████████████████████████████
  implementation                :  3026  ██████████████████████████████
  dp                            :  2496  ████████████████████████
  constructive algorithms       :  2091  ████████████████████
  data structures               :  2018  ████████████████████
  brute force                   :  1977  ███████████████████
  binary search                 :  1284  ████████████
  sortings                      :  1233  ████████████
  graphs                        :  1218  ████████████
  dfs and similar               :  1065  ██████████
  trees                         :   961  █████████
  number theory                 :   882  ████████
  combinatorics                 :   830  ████████
  strings                       :   812  ███

In [3]:
import pandas as pd

data = []
for p in problems:
    if p.rating and p.rating > 0 and p.tags:
        data.append({
            'problem_id': p.problem_id,
            'title':      p.title,
            'rating':     p.rating,
            'tags':       p.tags
        })

df = pd.DataFrame(data)

print("Problems per rating bucket:")
buckets = [800, 1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 3500]
for i in range(len(buckets) - 1):
    low  = buckets[i]
    high = buckets[i+1]
    count = len(df[(df['rating'] >= low) & (df['rating'] < high)])
    bar  = '█' * (count // 50)
    print(f"  {low}-{high}: {count:5d}  {bar}")

Problems per rating bucket:
  800-1000:  1420  ████████████████████████████
  1000-1200:   830  ████████████████
  1200-1400:   913  ██████████████████
  1400-1600:   924  ██████████████████
  1600-1800:  1021  ████████████████████
  1800-2000:  1012  ████████████████████
  2000-2200:   927  ██████████████████
  2200-2400:   867  █████████████████
  2400-3500:  2682  █████████████████████████████████████████████████████


In [4]:
# Simulate: user is rated 1350, weak in graphs, comfort rating 1280
comfort_rating = 1280
weak_tag       = "graphs"

low  = comfort_rating + 100   # 1380
high = comfort_rating + 200   # 1480

# Filter problems
candidates = df[
    (df['rating'] >= low) &
    (df['rating'] <= high) &
    (df['tags'].apply(lambda tags: weak_tag in tags))
]

print(f"Problems in tag '{weak_tag}' rated {low}-{high}:")
print(f"Total candidates: {len(candidates)}")
print()
print(candidates[['problem_id', 'title', 'rating']].head(10).to_string())

Problems in tag 'graphs' rated 1380-1480:
Total candidates: 28

     problem_id                                  title  rating
195       2204D                       Alternating Path    1400
654       2133C                             The Nether    1400
669       2131D                     Arboris Contractio    1400
1029      2066A                  Object Identification    1400
1273      2034C       Trapped in the Witch's Labyrinth    1400
1278      2033E  Sakurako, Kosuke, and the Permutation    1400
2123      1893A                    Anonymous Informant    1400
2476      1830A                Copil Copac Draws Trees    1400
2630     1800E1      Unforgivable Curse (easy version)    1400
2882      1764C             Doremy's City Construction    1400


In [6]:
tags_to_test = ['dp', 'graphs', 'greedy', 'binary search', 'dsu']

print("Available problems per tag at 1200-1400 rating:")
for tag in tags_to_test:
    count = len(df[
        (df['rating'] >= 1200) &
        (df['rating'] <= 1400) &
        (df['tags'].apply(lambda tags: tag in tags))
    ])
    status = "✅" if count > 5 else "⚠️"
    print(f"  {status} {tag:20s}: {count} problems")

Available problems per tag at 1200-1400 rating:
  ✅ dp                  : 153 problems
  ✅ graphs              : 64 problems
  ✅ greedy              : 583 problems
  ✅ binary search       : 147 problems
  ✅ dsu                 : 32 problems
